# Bag-of-words, and what one word of context buys

Throw word order away entirely and get 88% on IMDB. Then add bigrams and get 90%. Both numbers are more interesting than they look.

**Runs on:** CPU — about 3 minutes &nbsp;·&nbsp; **Slides:** [Chapter 14 — Text Classification](../../../course-web-slides/ch14/index.html) &nbsp;·&nbsp; **Section:** 02 — Sets: the bag-of-words approach

---

## The data, as raw text

In [ ]:
import os, pathlib, shutil, random
import keras

# The book downloads aclImdb; adjust the path if you already have it.
base_dir = pathlib.Path("aclImdb")
if not base_dir.exists():
    zip_path = keras.utils.get_file(
        origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
        fname="imdb", extract=True)
    base_dir = pathlib.Path(zip_path) / "aclImdb"
    shutil.rmtree(base_dir / "train" / "unsup", ignore_errors=True)

val_dir = base_dir / "val"
train_dir = base_dir / "train"
if not val_dir.exists():
    for category in ("neg", "pos"):
        os.makedirs(val_dir / category)
        files = os.listdir(train_dir / category)
        random.Random(1337).shuffle(files)
        num_val_samples = int(0.2 * len(files))
        for fname in files[-num_val_samples:]:
            shutil.move(train_dir / category / fname, val_dir / category / fname)

from keras.utils import text_dataset_from_directory
batch_size = 32
train_ds = text_dataset_from_directory(base_dir / "train", batch_size=batch_size)
val_ds = text_dataset_from_directory(base_dir / "val", batch_size=batch_size)
test_ds = text_dataset_from_directory(base_dir / "test", batch_size=batch_size)

for inputs, targets in train_ds:
    print("inputs.shape:", inputs.shape, inputs.dtype)
    print("first review:", inputs[0].numpy()[:200], "...")
    print("label:", targets[0].numpy())
    break

## Unigrams, multi-hot

In [ ]:
from keras import layers

text_vectorization = layers.TextVectorization(
    max_tokens=20000, output_mode="multi_hot")

text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)      # training split only

binary_1gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
binary_1gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
binary_1gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)

for inputs, targets in binary_1gram_train_ds:
    print("inputs.shape:", inputs.shape)
    print("a single review:", inputs[0].numpy()[:20], "...")
    break

A 20,000-vector of ones and zeros. **Word order is gone completely** — *dog bites man* and *man bites dog* are the same input.

## The model, and a number to remember

In [ ]:
def get_model(max_tokens=20000, hidden_dim=16):
    inputs = keras.Input(shape=(max_tokens,))
    x = layers.Dense(hidden_dim, activation="relu")(inputs)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop", loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

model = get_model()
cb = [keras.callbacks.ModelCheckpoint("binary_1gram.keras",
                                      save_best_only=True)]
model.fit(binary_1gram_train_ds.cache(),
          validation_data=binary_1gram_val_ds.cache(),
          epochs=10, callbacks=cb, verbose=2)

model = keras.models.load_model("binary_1gram.keras")
acc_1gram = model.evaluate(binary_1gram_test_ds, verbose=0)[1]
print(f"\nunigram test accuracy: {acc_1gram:.3f}")

Expected output:

```
unigram test accuracy: 0.88x
```

**Eighty-eight percent, with no word order at all.** That number is worth sitting with. Sentiment is largely carried by *which words appear*, and a great deal of what looks like language understanding is vocabulary statistics.

## Bigrams

In [ ]:
text_vectorization = layers.TextVectorization(
    ngrams=2, max_tokens=20000, output_mode="multi_hot")
text_vectorization.adapt(text_only_train_ds)

binary_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
binary_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
binary_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)

model = get_model()
cb = [keras.callbacks.ModelCheckpoint("binary_2gram.keras",
                                      save_best_only=True)]
model.fit(binary_2gram_train_ds.cache(),
          validation_data=binary_2gram_val_ds.cache(),
          epochs=10, callbacks=cb, verbose=2)

model = keras.models.load_model("binary_2gram.keras")
acc_2gram = model.evaluate(binary_2gram_test_ds, verbose=0)[1]
print(f"\nbigram test accuracy: {acc_2gram:.3f}  "
      f"(+{acc_2gram - acc_1gram:.3f})")

Expected output:

```
bigram test accuracy: 0.90x  (+0.02x)
```

Two points, from **one word of context**. `"not good"` is now a feature in its own right rather than *not* plus *good*.

That is the entire argument for local order, and it is worth noting how small it is — which is what makes the sequence models in the next notebook a harder sell than they first appear.

## TF-IDF

In [ ]:
text_vectorization = layers.TextVectorization(
    ngrams=2, max_tokens=20000, output_mode="tf_idf")
text_vectorization.adapt(text_only_train_ds)

tfidf_train_ds = train_ds.map(lambda x, y: (text_vectorization(x), y),
                              num_parallel_calls=4)
tfidf_val_ds = val_ds.map(lambda x, y: (text_vectorization(x), y),
                          num_parallel_calls=4)
tfidf_test_ds = test_ds.map(lambda x, y: (text_vectorization(x), y),
                            num_parallel_calls=4)

model = get_model()
cb = [keras.callbacks.ModelCheckpoint("tfidf_2gram.keras",
                                      save_best_only=True)]
model.fit(tfidf_train_ds.cache(), validation_data=tfidf_val_ds.cache(),
          epochs=10, callbacks=cb, verbose=2)
acc_tfidf = keras.models.load_model("tfidf_2gram.keras").evaluate(
    tfidf_test_ds, verbose=0)[1]
print(f"\nTF-IDF bigram test accuracy: {acc_tfidf:.3f}")

**Term frequency divided by document frequency**: a word that appears in every review carries no signal; one that appears in this review and few others does. TF-IDF is decades older than deep learning and still competitive on short texts.

## Which words the model relies on

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tv = layers.TextVectorization(max_tokens=20000, output_mode="multi_hot")
tv.adapt(text_only_train_ds)
vocab = tv.get_vocabulary()

m = keras.models.load_model("binary_1gram.keras")
W1 = m.layers[1].get_weights()[0]      # (20000, 16)
W2 = m.layers[3].get_weights()[0]      # (16, 1)
influence = (W1 @ W2).ravel()

order = influence.argsort()
print("most negative words:")
for i in order[:12]:
    print(f"  {vocab[i]:16s} {influence[i]:+.3f}")
print("\nmost positive words:")
for i in order[::-1][:12]:
    print(f"  {vocab[i]:16s} {influence[i]:+.3f}")

A crude linearization — the network is not linear — but informative. Expect *worst*, *waste*, *awful* at one end and *excellent*, *perfect*, *wonderful* at the other. **If you see something surprising there, look at it**: that is where dataset artifacts hide.

---

## What to take away

- Bag-of-words discards order entirely and still reaches 88% on sentiment.
- Bigrams add one word of context for two points — a small, real gain.
- TF-IDF downweights words that appear everywhere; still competitive on short texts.
- Inspecting the most influential words is cheap and finds dataset artifacts.